In [1]:
import pandas as pd
import sqlite3

# Leer el dataset real del ISSSTE
df = pd.read_csv('../data/inventario_issste.csv')

print(f"✅ Archivo cargado correctamente")
print(f"   → Filas: {len(df):,}")
print(f"   → Columnas: {df.columns.tolist()}")
print(f"   → Rango de fechas: {df['fecha_corte'].min()} a {df['fecha_corte'].max()}")
print(f"   → Productos únicos: {df['clave_insumo'].nunique():,}")
print(f"   → Grupos terapéuticos: {df['grupo_terapeutico'].nunique():,}")
print()
print(df.head(3))

✅ Archivo cargado correctamente
   → Filas: 26,750
   → Columnas: ['tipo_insumo', 'clave_insumo', 'descripcion', 'grupo_terapeutico', 'demanda_mensual_nacional', 'inventario_piezas', 'fecha_corte']
   → Rango de fechas: 2025-04-01 a 2025-04-30
   → Productos únicos: 966
   → Grupos terapéuticos: 54

   tipo_insumo  clave_insumo  \
0  medicamento   10000002200   
1  medicamento   10000002200   
2  medicamento   10000002200   

                                         descripcion grupo_terapeutico  \
0  caseinato de calcio. polvo. cada 100 g contien...       nutriología   
1  caseinato de calcio. polvo. cada 100 g contien...       nutriología   
2  caseinato de calcio. polvo. cada 100 g contien...       nutriología   

   demanda_mensual_nacional  inventario_piezas fecha_corte  
0                     12478              24086  2025-04-01  
1                     12478              23273  2025-04-02  
2                     12478              23148  2025-04-03  


In [2]:
# ── 1. Verificar valores nulos ──────────────────
print("🔍 Valores nulos por columna:")
print(df.isnull().sum())
print()

# ── 2. Verificar duplicados ─────────────────────
duplicados = df.duplicated().sum()
print(f"🔍 Registros duplicados: {duplicados}")
print()

# ── 3. Limpiar datos ────────────────────────────
df['descripcion'] = df['descripcion'].str.strip().str.lower()
df['tipo_insumo'] = df['tipo_insumo'].str.strip().str.lower()
df['grupo_terapeutico'] = df['grupo_terapeutico'].str.strip().str.lower()
df['fecha_corte'] = pd.to_datetime(df['fecha_corte'])

# ── 4. Eliminar nulos si existen ────────────────
df = df.dropna(subset=['clave_insumo', 'inventario_piezas', 'demanda_mensual_nacional'])

print(f"✅ Limpieza completada")
print(f"   → Registros finales: {len(df):,}")
print(f"   → Tipos de datos:")
print(df.dtypes)

🔍 Valores nulos por columna:
tipo_insumo                 0
clave_insumo                0
descripcion                 0
grupo_terapeutico           0
demanda_mensual_nacional    0
inventario_piezas           0
fecha_corte                 0
dtype: int64

🔍 Registros duplicados: 0

✅ Limpieza completada
   → Registros finales: 26,750
   → Tipos de datos:
tipo_insumo                         object
clave_insumo                         int64
descripcion                         object
grupo_terapeutico                   object
demanda_mensual_nacional             int64
inventario_piezas                    int64
fecha_corte                 datetime64[ns]
dtype: object


In [3]:
# ── 1. Calcular días de cobertura ───────────────
# ¿Cuántos días dura el inventario actual a ritmo de demanda?
df['dias_cobertura'] = (df['inventario_piezas'] / (df['demanda_mensual_nacional'] / 30)).round(2)

# ── 2. Clasificar estatus de stock ──────────────
def clasificar_stock(dias):
    if dias <= 7:
        return 'critico'      # 🔴 menos de 1 semana
    elif dias <= 15:
        return 'bajo'         # 🟡 menos de 2 semanas
    elif dias <= 30:
        return 'normal'       # 🟢 menos de 1 mes
    else:
        return 'optimo'       # 🔵 más de 1 mes

df['estatus_stock'] = df['dias_cobertura'].apply(clasificar_stock)

# ── 3. Resumen de estatus ───────────────────────
print("📊 Distribución de estatus de stock:")
print(df['estatus_stock'].value_counts())
print()

# ── 4. Productos críticos hoy (último día) ──────
ultimo_dia = df['fecha_corte'].max()
df_hoy = df[df['fecha_corte'] == ultimo_dia]
criticos = df_hoy[df_hoy['estatus_stock'] == 'critico']

print(f"📅 Fecha de corte más reciente: {ultimo_dia.date()}")
print(f"🔴 Productos en estado CRÍTICO: {len(criticos)}")
print()
print("Top 10 productos más críticos:")
print(criticos[['descripcion', 'grupo_terapeutico', 
                'inventario_piezas', 'dias_cobertura']]
      .sort_values('dias_cobertura')
      .head(10)
      .to_string())

📊 Distribución de estatus de stock:
estatus_stock
optimo     23485
normal      1801
critico      795
bajo         669
Name: count, dtype: int64

📅 Fecha de corte más reciente: 2025-04-30
🔴 Productos en estado CRÍTICO: 21

Top 10 productos más críticos:
                                                                                                                                                                                                                                                                                                                                              descripcion             grupo_terapeutico  inventario_piezas  dias_cobertura
17077                                                                                                                                                                                                                              nalbufina solución inyectable cada ampolleta contiene: clorhidrato de nalbufina 10 mg envase con 5 ampolletas

In [4]:
# Conectar a la base de datos
conn = sqlite3.connect('../database/erillam.db')

# ── 1. Cargar catálogo de productos ─────────────
productos = df[['clave_insumo', 'descripcion', 
                'tipo_insumo', 'grupo_terapeutico']].drop_duplicates(
                subset='clave_insumo')

productos.to_sql('productos', conn, 
                 if_exists='replace', 
                 index=False)

print(f"✅ Productos cargados: {len(productos):,}")

# ── 2. Cargar inventario diario ──────────────────
inventario = df[['clave_insumo', 'fecha_corte', 
                 'inventario_piezas', 
                 'demanda_mensual_nacional',
                 'dias_cobertura', 
                 'estatus_stock']].copy()

inventario['fecha_corte'] = inventario['fecha_corte'].astype(str)

inventario.to_sql('inventario_diario', conn, 
                  if_exists='replace', 
                  index=False)

print(f"✅ Inventario diario cargado: {len(inventario):,} registros")

# ── 3. Cargar alertas de desabasto ───────────────
alertas = df[df['estatus_stock'].isin(['critico', 'bajo'])][
    ['clave_insumo', 'fecha_corte', 
     'estatus_stock', 'dias_cobertura']].copy()

alertas['fecha_corte'] = alertas['fecha_corte'].astype(str)
alertas['mensaje'] = alertas.apply(
    lambda r: f"Stock crítico: {r['dias_cobertura']} días de cobertura" 
    if r['estatus_stock'] == 'critico' 
    else f"Stock bajo: {r['dias_cobertura']} días de cobertura", axis=1)

alertas.rename(columns={'estatus_stock': 'tipo_alerta',
                         'fecha_corte': 'fecha_alerta'}, inplace=True)

alertas.to_sql('alertas_desabasto', conn, 
               if_exists='replace', 
               index=False)

print(f"✅ Alertas cargadas: {len(alertas):,} registros")

# ── 4. Cargar resumen por grupos ─────────────────
resumen = df.groupby(['grupo_terapeutico', 'fecha_corte']).agg(
    total_productos=('clave_insumo', 'nunique'),
    productos_criticos=('estatus_stock', lambda x: (x == 'critico').sum()),
    productos_bajo_stock=('estatus_stock', lambda x: (x == 'bajo').sum()),
    inventario_total=('inventario_piezas', 'sum'),
    demanda_total=('demanda_mensual_nacional', 'sum')
).reset_index()

resumen['fecha_corte'] = resumen['fecha_corte'].astype(str)

resumen.to_sql('resumen_grupos', conn, 
               if_exists='replace', 
               index=False)

print(f"✅ Resumen por grupos cargado: {len(resumen):,} registros")

conn.close()
print()
print("🎉 ETL completado — Base de datos lista")

✅ Productos cargados: 966
✅ Inventario diario cargado: 26,750 registros
✅ Alertas cargadas: 1,464 registros
✅ Resumen por grupos cargado: 1,609 registros

🎉 ETL completado — Base de datos lista


In [5]:
conn = sqlite3.connect('../database/erillam.db')

tablas = ['productos', 'inventario_diario', 
          'alertas_desabasto', 'resumen_grupos']

print("📊 Verificación final de la base de datos:")
print("─" * 45)
for tabla in tablas:
    total = pd.read_sql(f"SELECT COUNT(*) as total FROM {tabla}", 
                        conn).iloc[0]['total']
    print(f"   → {tabla:<25} {total:>7,} registros")

print("─" * 45)
print()

# Consulta de negocio: top 5 grupos con más productos críticos hoy
print("🔴 Top 5 grupos con más productos críticos hoy:")
query = """
    SELECT 
        grupo_terapeutico,
        productos_criticos,
        total_productos,
        ROUND(productos_criticos * 100.0 / total_productos, 1) as pct_criticos
    FROM resumen_grupos
    WHERE fecha_corte = '2025-04-30'
    AND productos_criticos > 0
    ORDER BY productos_criticos DESC
    LIMIT 5
"""
print(pd.read_sql(query, conn).to_string(index=False))
conn.close()

📊 Verificación final de la base de datos:
─────────────────────────────────────────────
   → productos                     966 registros
   → inventario_diario          26,750 registros
   → alertas_desabasto           1,464 registros
   → resumen_grupos              1,609 registros
─────────────────────────────────────────────

🔴 Top 5 grupos con más productos críticos hoy:
    grupo_terapeutico  productos_criticos  total_productos  pct_criticos
       anestesiología                   2                8          25.0
médicas y quirúrgicas                   2              151           1.3
           neurología                   2               44           4.5
            oncología                   2              116           1.7
            analgesia                   1               21           4.8
